# GroundLie360 — Análisis de experimentos (E1, E2, E3, E5, E6, E6b)


In [ ]:
import sqlite3, json
from pathlib import Path
import numpy as np
import pandas as pd

# BASE = carpeta de este notebook (experiments/GroundLie360). Ajustar si se corre desde otro cwd.
BASE = Path.cwd()
if (BASE / 'experiments' / 'GroundLie360').exists():
    BASE = BASE / 'experiments' / 'GroundLie360'

DBS = {
    'qwen2b': BASE / 'open_source/results/qwen3vl_2b/qwen3vl_cache.db',
    'qwen8b': BASE / 'open_source/results/qwen3vl_8b/qwen3vl_cache.db',
    'wave':   BASE / 'open_source/results/WAVE7B/wave_cache.db',
    'ge2':    BASE / 'google_embeddings2/results/groundlie_ge2.db',
}
# E2/E3 de GE2 viven en una DB separada (paralelización). Globales de GE2 en la de arriba.
GE2_SEGMENTS_DB = BASE / 'google_embeddings2/results/groundlie_ge2_segments.db'
GE2_SUBSET_JSON = BASE / 'e3_ge2_subset.json'

MODELS = list(DBS)
for m, p in DBS.items():
    print(f'{m:8s} {"OK" if p.exists() else "FALTA":5s} {p}')

In [ ]:
# ---- helpers ----
def connect(path):
    return sqlite3.connect(str(path))

def load_globals(conn, modality):
    """raw_id -> np.float32 vector para una modalidad de la tabla embeddings."""
    rows = conn.execute('SELECT raw_id, vector FROM embeddings WHERE modality=?', (modality,)).fetchall()
    return {r[0]: np.frombuffer(r[1], dtype=np.float32) for r in rows}

def load_segments(conn, segment_type, modality):
    """raw_id -> list de dicts {segment_idx,start_s,end_s,vector,extra} ordenados."""
    rows = conn.execute(
        'SELECT raw_id, segment_idx, start_s, end_s, vector, extra_json FROM segment_embeddings '
        'WHERE segment_type=? AND modality=? ORDER BY raw_id, segment_idx', (segment_type, modality)).fetchall()
    out = {}
    for rid, idx, s0, s1, blob, extra in rows:
        out.setdefault(rid, []).append({'segment_idx': idx, 'start_s': s0, 'end_s': s1,
                                        'vector': np.frombuffer(blob, dtype=np.float32),
                                        'extra': json.loads(extra) if extra else None})
    return out

def load_labels(conn):
    return pd.read_sql_query('SELECT * FROM groundlie_labels', conn).set_index('raw_id')

def cos(a, b):
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    if na == 0 or nb == 0:
        return np.nan
    return float(np.dot(a, b) / (na * nb))

def auc_fake(score, is_fake):
    """AUC de 'score' separando fake(1) vs real(0). score alto -> mas fake."""
    s = np.asarray(score, float); y = np.asarray(is_fake, int)
    ok = ~np.isnan(s)
    s, y = s[ok], y[ok]
    if len(np.unique(y)) < 2:
        return np.nan
    order = s.argsort()
    ranks = np.empty_like(order, float); ranks[order] = np.arange(1, len(s) + 1)
    n1 = y.sum(); n0 = len(y) - n1
    return (ranks[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)

## E1 — cos(video_global, título)


In [ ]:
e1 = {}
for m in MODELS:
    c = connect(DBS[m]); lab = load_labels(c)
    vid, tit = load_globals(c, 'video'), load_globals(c, 'text_title')
    recs = []
    for rid in set(vid) & set(tit) & set(lab.index):
        recs.append({'raw_id': rid, 'cos_vt': cos(vid[rid], tit[rid]),
                     'binary_label': lab.loc[rid, 'binary_label'],
                     'false_title': lab.loc[rid, 'false_title']})
    df = pd.DataFrame(recs); e1[m] = df
    real = df[df.binary_label == 0].cos_vt.mean(); fake = df[df.binary_label == 1].cos_vt.mean()
    print(f'{m:8s} N={len(df):4d}  cos real={real:.4f}  fake={fake:.4f}  Δ={real-fake:+.4f}  '
          f'AUC(-cos)={auc_fake(-df.cos_vt, df.binary_label):.3f}')
    c.close()

## E2 — ventana textual (False Speech)


In [ ]:
e2 = {}
for m in MODELS:
    c = connect(DBS[m])
    seg_db = connect(GE2_SEGMENTS_DB) if m == 'ge2' else c
    tit = load_globals(c, 'text_title')
    fake = load_segments(seg_db, 'window_fake', 'transcript')
    clean = load_segments(seg_db, 'window_clean', 'transcript')
    recs = []
    for rid in set(fake) & set(clean) & set(tit):
        cf = cos(fake[rid][0]['vector'], tit[rid])
        cc = cos(clean[rid][0]['vector'], tit[rid])
        recs.append({'raw_id': rid, 'cos_fake': cf, 'cos_clean': cc, 'delta': cc - cf})
    df = pd.DataFrame(recs); e2[m] = df
    print(f'{m:8s} N={len(df):3d}  Δ medio (clean-fake)={df.delta.mean():+.4f}  '
          f'% Δ>0={100*(df.delta>0).mean():.0f}%')
    if m == 'ge2': seg_db.close()
    c.close()

## E3 — coherencia entre escenas (Temporal Edit)


In [ ]:
e3 = {}
for m in MODELS:
    c = connect(DBS[m]); lab = load_labels(c)
    seg_db = connect(GE2_SEGMENTS_DB) if m == 'ge2' else c
    scenes = load_segments(seg_db, 'scene', 'video')
    recs = []
    for rid, segs in scenes.items():
        if len(segs) < 2 or rid not in lab.index:
            continue
        adj = [cos(segs[i]['vector'], segs[i+1]['vector']) for i in range(len(segs)-1)]
        recs.append({'raw_id': rid, 'mean_adj_cos': np.nanmean(adj), 'n_scenes': len(segs),
                     'temporal_edit': lab.loc[rid, 'temporal_edit'],
                     'binary_label': lab.loc[rid, 'binary_label']})
    df = pd.DataFrame(recs); e3[m] = df
    te1 = df[df.temporal_edit == 1].mean_adj_cos.mean(); te0 = df[df.temporal_edit == 0].mean_adj_cos.mean()
    print(f'{m:8s} N={len(df):4d}  adj-cos TE0={te0:.4f}  TE1={te1:.4f}  Δ(TE0-TE1)={te0-te1:+.4f}  '
          f'AUC(-cos|TE)={auc_fake(-df.mean_adj_cos, df.temporal_edit):.3f}')
    if m == 'ge2': seg_db.close()
    c.close()

## E5 — doble engaño


In [ ]:
# Usa qwen8b como referencia (cambiar a gusto). Tabla cos medio por (false_title, false_speech).
REF = 'qwen8b'
c = connect(DBS[REF]); lab = load_labels(c)
df = e1[REF].set_index('raw_id').join(lab[['false_speech', 'temporal_edit', 'cgi']])
print('cos(título,vídeo) medio por combinación de engaños [', REF, ']')
print(df.groupby(['false_title', 'false_speech']).cos_vt.agg(['mean', 'count']))
print()
print('por (temporal_edit, false_title):')
print(df.groupby(['temporal_edit', 'false_title']).cos_vt.agg(['mean', 'count']))
c.close()

## E6 — simplex V·A·T (WAVE only)


In [ ]:
c = connect(DBS['wave']); lab = load_labels(c)
V, A, T = load_globals(c, 'video'), load_globals(c, 'audio'), load_globals(c, 'transcript')
recs = []
for rid in set(V) & set(A) & set(T) & set(lab.index):
    d_va, d_vt, d_at = 1-cos(V[rid], A[rid]), 1-cos(V[rid], T[rid]), 1-cos(A[rid], T[rid])
    recs.append({'raw_id': rid, 'perim': d_va+d_vt+d_at, 'binary_label': lab.loc[rid, 'binary_label']})
e6 = pd.DataFrame(recs)
real, fake = e6[e6.binary_label==0].perim.mean(), e6[e6.binary_label==1].perim.mean()
print(f'WAVE V·A·T  N={len(e6)}  perim real={real:.4f}  fake={fake:.4f}  Δ={fake-real:+.4f}  '
      f'AUC(perim)={auc_fake(e6.perim, e6.binary_label):.3f}')
c.close()

## E6b — triángulo de coherencia título·video·transcript (4 modelos)


In [ ]:
def tri_area(a, b, c):
    s = (a + b + c) / 2
    return float(np.sqrt(max(s*(s-a)*(s-b)*(s-c), 0.0)))

e6b = {}
for m in MODELS:
    conn = connect(DBS[m]); lab = load_labels(conn)
    Ti, Vi, Tr = load_globals(conn, 'text_title'), load_globals(conn, 'video'), load_globals(conn, 'transcript')
    recs = []
    for rid in set(Ti) & set(Vi) & set(Tr) & set(lab.index):
        d_tv = 1-cos(Ti[rid], Vi[rid]); d_tt = 1-cos(Ti[rid], Tr[rid]); d_vt = 1-cos(Vi[rid], Tr[rid])
        recs.append({'raw_id': rid, 'd_title_video': d_tv, 'd_title_transcript': d_tt,
                     'd_video_transcript': d_vt, 'perim': d_tv+d_tt+d_vt,
                     'area': tri_area(d_tv, d_tt, d_vt),
                     'binary_label': lab.loc[rid, 'binary_label'],
                     'false_title': lab.loc[rid, 'false_title'],
                     'false_speech': lab.loc[rid, 'false_speech']})
    df = pd.DataFrame(recs); e6b[m] = df
    real, fake = df[df.binary_label==0].perim.mean(), df[df.binary_label==1].perim.mean()
    print(f'{m:8s} N={len(df):4d} (con transcript)  perim real={real:.4f} fake={fake:.4f} Δ={fake-real:+.4f}  '
          f'AUC(perim)={auc_fake(df.perim, df.binary_label):.3f}  AUC(area)={auc_fake(df.area, df.binary_label):.3f}')
    conn.close()

In [ ]:
# E6b — desglose por arista y por tipo de engaño (referencia qwen8b)
df = e6b['qwen8b']
print('AUC por arista (score=distancia, alto=incoherente=fake):')
for col in ['d_title_video', 'd_title_transcript', 'd_video_transcript', 'perim', 'area']:
    print(f'  {col:20s} AUC={auc_fake(df[col], df.binary_label):.3f}')
print('\nArista title-transcript media por false_speech:')
print(df.groupby('false_speech').d_title_transcript.agg(['mean', 'count']))
print('\nArista title-video media por false_title:')
print(df.groupby('false_title').d_title_video.agg(['mean', 'count']))